In [1]:
import pandas as pd
import math
from openpyxl import Workbook
from zipfile import ZipFile

In [2]:
order_report = pd.read_excel("Company ShopX - Order Report.xlsx")
product_weight = pd.read_excel("Company ShopX - Product Weight.xlsx")
pincode_zone = pd.read_excel("Company ShopX - Warehouse&Customer Pin Code and Zone details.xlsx")
courier_invoice = pd.read_excel("Courier Company - Invoice.xlsx")
courier_rates = pd.read_excel("Courier Company - Rates.xlsx")

In [3]:
merged = pd.merge(order_report, product_weight, on="Product Code")
merged["Total Weight (g)"] = merged["Units Ordered"] * merged["Product Weight (g)"]

shopx_weight = merged.groupby("Order ID")["Total Weight (g)"].sum().reset_index()
shopx_weight["Total weight as per ShopX (KG)"] = shopx_weight["Total Weight (g)"] / 1000

In [4]:
def weight_slab(x):
    return math.ceil(x / 0.5) * 0.5

shopx_weight["Weight slab as per ShopX (KG)"] = shopx_weight["Total weight as per ShopX (KG)"].apply(weight_slab)

In [5]:
zone_data = pincode_zone.rename(columns={"Delivery Zone": "Delivery Zone as per ShopX"})

zone_merge = pd.merge(
    courier_invoice[["Order ID","Store House Pincode","Customer Area Code"]],
    zone_data,
    on=["Store House Pincode","Customer Area Code"],
    how="left"
)


In [6]:
final = pd.merge(courier_invoice, shopx_weight, on="Order ID")
final = pd.merge(final, zone_merge[["Order ID","Delivery Zone as per ShopX"]], on="Order ID")

In [7]:
def calculate_expected(row):
    slab = row["Weight slab as per ShopX (KG)"]
    zone = row["Delivery Zone as per ShopX"].lower()
    freight = row["Freight Type"]

    fwd_fixed = courier_rates[f"fwd_{zone}_fixed"][0]
    fwd_add = courier_rates[f"fwd_{zone}_additional"][0]

    total = fwd_fixed

    if slab > 0.5:
        extra = slab - 0.5
        total += (extra / 0.5) * fwd_add

    if freight == "Forward and RTO charges":
        rto_fixed = courier_rates[f"rto_{zone}_fixed"][0]
        rto_add = courier_rates[f"rto_{zone}_additional"][0]

        total += rto_fixed

        if slab > 0.5:
            total += (extra / 0.5) * rto_add

    return total
final["Expected Charge as per ShopX (Rs.)"] = final.apply(calculate_expected, axis=1)

In [8]:
final["Difference Between Expected Charges and Billed Charges (Rs.)"] = \
    final["Expected Charge as per ShopX (Rs.)"] - final["Total Amount (Rs.)"]

In [9]:
final = final.rename(columns={"AWB Code (Airway Bill Number) ": "AWB Number"})

In [10]:
order_level = final[[
    "Order ID",
    "AWB Number",
    "Total weight as per ShopX (KG)",
    "Weight slab as per ShopX (KG)",
    "Chargeable Weight",
    "Delivery Zone as per ShopX",
    "Delivery Zone",
    "Expected Charge as per ShopX (Rs.)",
    "Total Amount (Rs.)",
    "Difference Between Expected Charges and Billed Charges (Rs.)"
]]

In [11]:
correct = order_level[order_level["Difference Between Expected Charges and Billed Charges (Rs.)"] == 0]
over = order_level[order_level["Difference Between Expected Charges and Billed Charges (Rs.)"] > 0]
under = order_level[order_level["Difference Between Expected Charges and Billed Charges (Rs.)"] < 0]

summary = [
    ["Correctly Charged", len(correct), correct["Total Amount (Rs.)"].sum()],
    ["Overcharged", len(over), over["Difference Between Expected Charges and Billed Charges (Rs.)"].sum()],
    ["Undercharged", len(under), under["Difference Between Expected Charges and Billed Charges (Rs.)"].sum()]
]

In [12]:

wb1 = Workbook()
ws1 = wb1.active
ws1.title = "Order Level Calculation"
ws1.append(list(order_level.columns))

for row in order_level.itertuples(index=False):
    ws1.append(list(row))

wb1.save("Order_Level_Result.xlsx")


wb2 = Workbook()
ws2 = wb2.active
ws2.title = "Summary"
ws2.append(["Description","Count","Amount (Rs.)"])

In [13]:
for row in summary:
    ws2.append(row)

wb2.save("Summary_Result.xlsx")

print("Project Completed Successfully")

Project Completed Successfully
